# 09 — Measuring State and Its Change: Scene State, Experiencer State, Choices and Influences

MallWorld state-dynamics redesign (2026-10)

---

## Why this notebook

The audited analyses (notebooks 02–07) treated each location as a scene to be averaged over all visitors, and each dream as a narrative. The state-dynamics design treats MallWorld as a common intermediate state that each visitor experiences through his or her own state:
- the living are, as to their spirit, already among spirits (*Heaven and Hell* §438);
- the world of spirits is an intermediate state in which people "are brought into one state after another" (§§421–427);
- "change of place is nothing else than change of state", likeness of state is nearness, and the way lengthens or shortens with desire (§§192–195);
- influence flows from above and from below through the sphere around each person (§§590–591).

The questions then concern **state and how it evolves**: who or what influences it, what arises from it, and how the person's choices go with its change. No observable is assigned to a degree.

This notebook is **measurement only**. It builds the state measures from several coded properties together, checks whether they behave as measures, and counts the transitions, choices and influences available for analysis. It does **not** estimate how choices or entities relate to state change. Those tests are registered separately before they are computed.

## Pre-specified scoring (fixed before any computation in this notebook)

Each indicator is scored −1 / 0 / +1. Categories whose valence is ambiguous are excluded (treated as missing), as is "not mentioned".

**Scene state** (what appears at the location):

| Indicator | +1 | 0 | −1 | Excluded |
|---|---|---|---|---|
| atmosphere | welcoming, peaceful | neutral | uncomfortable, oppressive, threatening, chaotic, eerie, wrong | nostalgic |
| light | bright_natural | bright_artificial | dim, flickering, dark, absent | colored, mixed |
| light_temperature | warm_golden | neutral | cold_white | — |
| cleanliness | clean | normal | dirty, filthy, disgusting | sterile |
| state (of place) | new, maintained | worn | decaying, abandoned, destroyed | under_construction, renovating, mixed |
| crowd_behavior | socializing | waiting, wandering | panic, mob, zombie_like | coordinated |
| water_presence | pool_clean, fountain, ocean_calm | river, rain | pool_dirty, flood, leak, ocean_turbulent, tsunami | none, submerged |

**Experiencer state** (the person at the location):

| Indicator | +1 | 0 | −1 | Excluded |
|---|---|---|---|---|
| affective_response | delight, comfort | curiosity, indifference, boredom | confusion, anxiety, disgust, horror | — |
| somatic_response | comfort | — | nausea, dizziness, paralysis, heaviness, headache, sleepiness | none, glitching, ejection |

A composite is the mean of the scored indicators available at a location.

**What would make a composite unusable** (decided now):
- the scene indicators do not hang together: median pairwise Spearman correlation among them ≤ 0.10, or any indicator with a negative item–rest correlation;
- a data-driven scaling (homogeneity analysis, below) orders the main categories against the a-priori scores (rank correlation ≤ 0.5 for atmosphere, light or cleanliness);
- the composite is driven by narrative length (|r| with log word count > 0.30).

## Known limits carried into every later analysis

- **Same source.** All indicators are coded by one model from one text, often one sentence.
- **Inference.** GPT-5.2 rated atmosphere about twice as often as a blind second coder, by inference (notebook 08). Where both rated, κ is 0.75 (atmosphere), 0.90 (light) and 0.67 (affect).
- **Priming.** The extraction prompt attached framework meanings to light temperature ("Charity/Good"), somatic distress ("Sphere Incompatibility"), affect ("Ruling Love") and transit mode ("Influx", "Reformation/Effort", "World of Spirits", "Rejection").

In [1]:
import sys, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, fisher_exact
from statsmodels.stats.multitest import multipletests

MW_ROOT = Path.cwd().parent
sys.path.insert(0, str(MW_ROOT / "scripts"))
import mallworld_dataset as mw  # verified loader: schema-checked fields, populations, cluster keys

OUTPUT_DIR = MW_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

posts = mw.load_posts()
primary = posts[posts["primary"]].copy()
print(f"Posts: {len(posts):,} | primary dream reports: {len(primary):,} from {primary['author'].nunique():,} authors")

Posts: 3,732 | primary dream reports: 1,918 from 1,303 authors


In [2]:
L = mw.load_locations()
C = mw.load_connections()
I = mw.load_interactions()
E = mw.load_entities()

SCENE = {
    "atmosphere": {"welcoming": 1, "peaceful": 1, "neutral": 0, "uncomfortable": -1, "oppressive": -1,
                   "threatening": -1, "chaotic": -1, "eerie": -1, "wrong": -1},
    "light": {"bright_natural": 1, "bright_artificial": 0, "dim": -1, "flickering": -1, "dark": -1, "absent": -1},
    "light_temperature": {"warm_golden": 1, "neutral": 0, "cold_white": -1},
    "cleanliness": {"clean": 1, "normal": 0, "dirty": -1, "filthy": -1, "disgusting": -1},
    "state": {"new": 1, "maintained": 1, "worn": 0, "decaying": -1, "abandoned": -1, "destroyed": -1},
    "crowd_behavior": {"socializing": 1, "waiting": 0, "wandering": 0, "panic": -1, "mob": -1, "zombie_like": -1},
    "water_presence": {"pool_clean": 1, "fountain": 1, "ocean_calm": 1, "river": 0, "rain": 0, "pool_dirty": -1,
                       "flood": -1, "leak": -1, "ocean_turbulent": -1, "tsunami": -1},
}
EXPER = {
    "affective_response": {"delight": 1, "comfort": 1, "curiosity": 0, "indifference": 0, "boredom": 0,
                           "confusion": -1, "anxiety": -1, "disgust": -1, "horror": -1},
    "somatic_response": {"comfort": 1, "nausea": -1, "dizziness": -1, "paralysis": -1, "heaviness": -1,
                         "headache": -1, "sleepiness": -1},
}
# Every category named in the scoring must be a schema value
for table in (SCENE, EXPER):
    for col, mapping in table.items():
        mw.check_values(col, mapping)

for col, mapping in {**SCENE, **EXPER}.items():
    L["s_" + col] = L[col].map(mapping).astype(float)

scene_cols = ["s_" + c for c in SCENE]
exp_cols = ["s_" + c for c in EXPER]
L["n_scene"] = L[scene_cols].notna().sum(axis=1)
L["n_exp"] = L[exp_cols].notna().sum(axis=1)
L["scene_state"] = L[scene_cols].mean(axis=1)
L["exp_state"] = L[exp_cols].mean(axis=1)
L["scene_state_noatm"] = L[[c for c in scene_cols if c != "s_atmosphere"]].mean(axis=1)

print(f"{len(L):,} locations in {L.post_id.nunique():,} dreams (primary population)\n")
print("Coverage of each scored indicator:")
for c in scene_cols + exp_cols:
    k = int(L[c].notna().sum())
    print(f"  {c[2:]:20s} {mw.fmt_pct(k, len(L))}")
print()
for name, n in (("scene", "n_scene"), ("experiencer", "n_exp")):
    for k in (1, 2, 3):
        print(f"  locations with >= {k} {name} indicators: {mw.fmt_pct(int((L[n] >= k).sum()), len(L))}")
print(f"  locations with both a scene and an experiencer state: "
      f"{mw.fmt_pct(int((L.scene_state.notna() & L.exp_state.notna()).sum()), len(L))}")

8,685 locations in 1,918 dreams (primary population)

Coverage of each scored indicator:
  atmosphere           4145/8685 = 47.7% [95% CI 46.7–48.8]
  light                1319/8685 = 15.2% [95% CI 14.4–16.0]
  light_temperature    717/8685 = 8.3% [95% CI 7.7–8.9]
  cleanliness          654/8685 = 7.5% [95% CI 7.0–8.1]
  state                2471/8685 = 28.5% [95% CI 27.5–29.4]
  crowd_behavior       805/8685 = 9.3% [95% CI 8.7–9.9]
  water_presence       798/8685 = 9.2% [95% CI 8.6–9.8]
  affective_response   3770/8685 = 43.4% [95% CI 42.4–44.5]
  somatic_response     141/8685 = 1.6% [95% CI 1.4–1.9]

  locations with >= 1 scene indicators: 5265/8685 = 60.6% [95% CI 59.6–61.6]
  locations with >= 2 scene indicators: 3073/8685 = 35.4% [95% CI 34.4–36.4]
  locations with >= 3 scene indicators: 1497/8685 = 17.2% [95% CI 16.5–18.0]
  locations with >= 1 experiencer indicators: 3790/8685 = 43.6% [95% CI 42.6–44.7]
  locations with >= 2 experiencer indicators: 121/8685 = 1.4% [95% CI 1.2–1.

## M0 — How the data were acquired: descriptors for the bias model

Self-selection, community vocabulary, onset and posting volume, from the posts and their raw text. The text patterns are fixed here and are crude: they flag mentions, not exposure.

In [3]:
import re
prim = posts[posts.primary].copy()
N = len(prim)
print("Primary dream reports by year:", prim.year.value_counts().sort_index().to_dict())
per_author = prim.groupby("author").size()
print(f"Authors: {len(per_author):,}; with one dream {mw.fmt_pct(int((per_author == 1).sum()), len(per_author))}; "
      f"with >= 2 dreams {int((per_author >= 2).sum())} authors ({int(per_author[per_author >= 2].sum())} dreams)")
rec = mw.isin(prim, "recurrence", ["recurring", "long_term_recurring"])
print(f"Reports describing a recurring visit (recurring or long-term recurring): {mw.fmt_pct(int(rec.sum()), N)}")

records = {r["_post_id"]: r for r in mw.load_records()}
shared = prim.post_id.map(lambda pid: bool(records[pid]["extraction"]["meta"].get("is_asking_if_shared")))
print(f"Reports asking whether others share the experience: {mw.fmt_pct(int(shared.sum()), N)}")

text = prim.post_id.map(mw.raw_text)
VOCAB = re.compile(r"\bmall ?world\b", re.I)
ONSET = re.compile(r"\b(since i was (a )?(kid|child|little|young|teen|teenager|\d+)|as a (kid|child)|when i was (a )?(kid|child|little|young)"
                   r"|for (many |several |\d+ )?years|my whole life|all my life|childhood)\b", re.I)
prim["mentions_vocab"] = text.str.contains(VOCAB)
prim["mentions_onset"] = text.str.contains(ONSET)
print(f"Reports using the community's name ('mall world'): {mw.fmt_pct(int(prim.mentions_vocab.sum()), N)}")
print(f"Reports mentioning childhood or long-standing onset: {mw.fmt_pct(int(prim.mentions_onset.sum()), N)}")

Primary dream reports by year: {2021: 27, 2022: 88, 2023: 321, 2024: 517, 2025: 894, 2026: 71}
Authors: 1,303; with one dream 1025/1303 = 78.7% [95% CI 76.4–80.8]; with >= 2 dreams 278 authors (893 dreams)
Reports describing a recurring visit (recurring or long-term recurring): 1317/1918 = 68.7% [95% CI 66.6–70.7]
Reports asking whether others share the experience: 937/1918 = 48.9% [95% CI 46.6–51.1]


Reports using the community's name ('mall world'): 820/1918 = 42.8% [95% CI 40.6–45.0]


Reports mentioning childhood or long-standing onset: 252/1918 = 13.1% [95% CI 11.7–14.7]


/tmp/ipykernel_1688/2859860166.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  prim["mentions_onset"] = text.str.contains(ONSET)


## M1 — Do the scene indicators hang together?

Pairwise Spearman correlations among the scored scene indicators, at the same location, over locations where both are scored. Item–rest correlation: each indicator against the mean of the other scene indicators available at that location.

In [4]:
rows = []
for i, a in enumerate(scene_cols):
    for b in scene_cols[i + 1:]:
        d = L[[a, b]].dropna()
        if len(d) >= 30 and d[a].nunique() > 1 and d[b].nunique() > 1:
            rows.append((a[2:], b[2:], len(d), spearmanr(d[a], d[b]).correlation))
pairs = pd.DataFrame(rows, columns=["item_a", "item_b", "n", "rho"])
print(pairs.sort_values("rho", ascending=False).to_string(index=False, float_format="%.3f"))
print(f"\nmedian pairwise rho = {pairs.rho.median():.3f} over {len(pairs)} pairs with n >= 30")

print("\nItem-rest correlations (locations with the item and at least one other scene item):")
for c in scene_cols:
    others = [o for o in scene_cols if o != c]
    rest = L[others].mean(axis=1)
    d = pd.DataFrame({"x": L[c], "r": rest}).dropna()
    print(f"  {c[2:]:18s} n = {len(d):5d}  rho = {spearmanr(d.x, d.r).correlation:+.3f}")

           item_a            item_b    n   rho
      cleanliness             state  558 0.860
      cleanliness    water_presence  126 0.789
       atmosphere       cleanliness  543 0.681
            light       cleanliness  293 0.617
       atmosphere             light 1043 0.541
            state    water_presence  326 0.436
       atmosphere    crowd_behavior  664 0.435
       atmosphere             state 1928 0.425
       atmosphere light_temperature  578 0.416
light_temperature    crowd_behavior  127 0.384
            light light_temperature  537 0.365
            light             state  746 0.363
            light    water_presence  192 0.362
   crowd_behavior    water_presence   89 0.343
       atmosphere    water_presence  432 0.328
      cleanliness    crowd_behavior  108 0.318
            state    crowd_behavior  393 0.250
light_temperature       cleanliness  231 0.158
light_temperature             state  522 0.126
            light    crowd_behavior  185 0.119
light_tempera

## M2 — Data-driven check of the scoring (homogeneity analysis)

Homogeneity analysis (multiple correspondence analysis with missing values treated passively) places every category on a common dimension using only which categories co-occur. It does not use the a-priori scores. If the first dimension orders the categories as the scoring does, the scoring is not an imposition. Locations with at least two scene indicators are used.

In [5]:
def homals(df, cols, p=2, iters=500, seed=0):
    # Gifi homogeneity analysis with missing passive; returns object scores, category quantifications, discrimination
    rng = np.random.default_rng(seed)
    G = {}
    for c in cols:
        cats = sorted(df[c].dropna().unique())
        G[c] = (cats, np.column_stack([(df[c] == k).to_numpy(float) for k in cats]))
    n = len(df)
    M = sum(g.sum(axis=1) for _, g in G.values())
    X = rng.standard_normal((n, p))
    def normalise(X):
        X = X - (M @ X) / M.sum()
        S = X.T @ (M[:, None] * X) / n
        w, V = np.linalg.eigh(S)
        return X @ V @ np.diag(w ** -0.5) @ V.T
    X = normalise(X)
    for _ in range(iters):
        Y = {c: (g.T @ X) / g.sum(axis=0)[:, None] for c, (_, g) in G.items()}
        Xn = sum(g @ Y[c] for c, (_, g) in G.items()) / M[:, None]
        Xn = normalise(Xn)
        if np.abs(Xn - X).max() < 1e-9:
            X = Xn
            break
        X = Xn
    Y = {c: (g.T @ X) / g.sum(axis=0)[:, None] for c, (_, g) in G.items()}
    disc = {c: [(g.sum(axis=0) * Y[c][:, s] ** 2).sum() / n for s in range(p)] for c, (_, g) in G.items()}
    return X, {c: pd.DataFrame(Y[c], index=G[c][0]) for c in cols}, pd.DataFrame(disc, index=[f"dim{s+1}" for s in range(p)]).T

raw_scene = list(SCENE)
H = L.loc[L.n_scene >= 2, raw_scene].copy()
for c in raw_scene:
    H.loc[~H[c].isin(SCENE[c].keys()), c] = np.nan
X, Q, disc = homals(H, raw_scene)
sign = 1 if Q["atmosphere"].loc["welcoming", 0] > Q["atmosphere"].loc["threatening", 0] else -1
print(f"{len(H):,} locations with >= 2 scene indicators\n")
print("Discrimination measures (share of each item's variance on each dimension):")
print(disc.to_string(float_format="%.3f"))
print(f"\nMean discrimination: dim1 {disc.dim1.mean():.3f}, dim2 {disc.dim2.mean():.3f}\n")
print("Dimension-1 quantification of each category against the a-priori score:")
for c in raw_scene:
    q = (sign * Q[c][0]).rename("quant").to_frame()
    q["a_priori"] = q.index.map(SCENE[c])
    q["n"] = q.index.map(H[c].value_counts())
    rho = spearmanr(q.a_priori, q.quant).correlation if q.a_priori.nunique() > 1 else np.nan
    print(f"  {c}: rank correlation with a-priori score = {rho:+.2f}")
    print("     " + "; ".join(f"{k} {v:+.2f} (n={int(m)})" for k, v, m in zip(q.index, q.quant, q.n)))

3,073 locations with >= 2 scene indicators

Discrimination measures (share of each item's variance on each dimension):
                   dim1  dim2
atmosphere        0.261 0.236
light             0.020 0.112
light_temperature 0.004 0.022
cleanliness       0.021 0.087
state             0.094 0.162
crowd_behavior    0.161 0.038
water_presence    0.023 0.031

Mean discrimination: dim1 0.084, dim2 0.098

Dimension-1 quantification of each category against the a-priori score:
  atmosphere: rank correlation with a-priori score = -0.17
     chaotic -1.14 (n=307); eerie +0.49 (n=488); neutral +0.10 (n=605); oppressive +0.12 (n=163); peaceful +0.12 (n=113); threatening -0.72 (n=395); uncomfortable +0.46 (n=317); welcoming -0.01 (n=299); wrong +0.21 (n=75)
  light: rank correlation with a-priori score = +0.44
     absent -0.25 (n=18); bright_artificial +0.07 (n=306); bright_natural +0.10 (n=311); dark +0.01 (n=283); dim +0.41 (n=276); flickering -0.75 (n=18)
  light_temperature: rank correlatio

## M2b — Post hoc: why the pre-specified scaling check failed

The check above was fixed before computation, and by its own criterion it fails: dimension 1 orders atmosphere and cleanliness against the a-priori scores. Two features of the output point to the cause. The first two dimensions have almost the same discrimination, so the solution does not identify which of them is "first"; and dimension 1 is dominated by rare extreme categories (tsunami n = 5, mob 16, flickering 18, destroyed 62). The cells below test that diagnosis. They were written after seeing the result and are labelled accordingly. **The a-priori scoring is not changed.**

In [6]:
print("1. Stability across random starts (p = 2): dimension-1 rank correlation with the a-priori scores")
for seed in range(6):
    Xs, Qs, ds = homals(H, raw_scene, p=2, seed=seed)
    sg = 1 if Qs["atmosphere"].loc["welcoming", 0] > Qs["atmosphere"].loc["threatening", 0] else -1
    r = {c: spearmanr(Qs[c].index.map(SCENE[c]), sg * Qs[c][0]).correlation for c in ("atmosphere", "light", "cleanliness")}
    print(f"   seed {seed}: discrimination dim1 {ds.dim1.mean():.3f} / dim2 {ds.dim2.mean():.3f} | "
          + "; ".join(f"{k} {v:+.2f}" for k, v in r.items()))

print("\n2. Categories with fewer than 40 locations set aside")
H2 = H.copy()
for c in raw_scene:
    vc = H2[c].value_counts()
    H2.loc[H2[c].isin(vc[vc < 40].index), c] = np.nan
H2 = H2[H2.notna().sum(axis=1) >= 2]
X2, Q2, d2 = homals(H2, raw_scene, p=2)
sg = 1 if Q2["atmosphere"].loc["welcoming", 0] > Q2["atmosphere"].loc["threatening", 0] else -1
print(f"   n = {len(H2):,}; discrimination dim1 {d2.dim1.mean():.3f} / dim2 {d2.dim2.mean():.3f}")
for c in raw_scene:
    a = Q2[c].index.map(SCENE[c])
    if a.nunique() > 1:
        print(f"   {c:18s} rank correlation with a-priori score = {spearmanr(a, sg * Q2[c][0]).correlation:+.2f}")

print("\n3. Category rest-score: mean a-priori score of the OTHER scene indicators at locations with each category")
for c in raw_scene:
    others = ["s_" + o for o in raw_scene if o != c]
    d = pd.DataFrame({"cat": L[c], "a": L[c].map(SCENE[c]), "rest": L[others].mean(axis=1)}).dropna()
    m = d.groupby("cat").agg(a=("a", "first"), rest=("rest", "mean"), n=("rest", "size"))
    print(f"   {c}: rank correlation = {spearmanr(m.a, m.rest).correlation:+.2f} | "
          + "; ".join(f"{k} {r:+.2f} (n={n})" for k, r, n in zip(m.index, m.rest, m.n)))

1. Stability across random starts (p = 2): dimension-1 rank correlation with the a-priori scores


   seed 0: discrimination dim1 0.084 / dim2 0.098 | atmosphere -0.17; light +0.44; cleanliness -0.89


   seed 1: discrimination dim1 0.092 / dim2 0.090 | atmosphere +0.84; light +0.85; cleanliness +0.89


   seed 2: discrimination dim1 0.093 / dim2 0.089 | atmosphere +0.84; light +0.85; cleanliness +0.89


   seed 3: discrimination dim1 0.083 / dim2 0.099 | atmosphere +0.00; light +0.44; cleanliness -0.22


   seed 4: discrimination dim1 0.091 / dim2 0.091 | atmosphere +0.54; light +0.85; cleanliness +0.89
   seed 5: discrimination dim1 0.087 / dim2 0.095 | atmosphere +0.84; light +0.85; cleanliness +0.89

2. Categories with fewer than 40 locations set aside


   n = 3,001; discrimination dim1 0.098 / dim2 0.083
   atmosphere         rank correlation with a-priori score = +0.84
   light              rank correlation with a-priori score = +0.95
   light_temperature  rank correlation with a-priori score = +1.00
   cleanliness        rank correlation with a-priori score = +1.00
   state              rank correlation with a-priori score = +0.80
   crowd_behavior     rank correlation with a-priori score = +0.63
   water_presence     rank correlation with a-priori score = +0.93

3. Category rest-score: mean a-priori score of the OTHER scene indicators at locations with each category
   atmosphere: rank correlation = +0.84 | chaotic +0.29 (n=307); eerie -0.23 (n=488); neutral +0.63 (n=605); oppressive -0.29 (n=163); peaceful +0.66 (n=113); threatening -0.29 (n=395); uncomfortable +0.01 (n=317); welcoming +0.82 (n=299); wrong +0.25 (n=75)
   light: rank correlation = +0.85 | absent -0.83 (n=18); bright_artificial +0.10 (n=306); bright_natural +0.48 

## M3 — The composites: distribution, length, and scene vs experiencer

In [7]:
for name in ("scene_state", "exp_state", "scene_state_noatm"):
    s = L[name].dropna()
    r = np.corrcoef(L.loc[s.index, "log_words"], s)[0, 1]
    print(f"{name:18s} n = {len(s):5d}  mean {s.mean():+.3f}  sd {s.std():.3f}  "
          f"share < 0: {(s < 0).mean():.1%}  share > 0: {(s > 0).mean():.1%}  r(log words) = {r:+.3f}")

both = L[["scene_state", "exp_state"]].dropna()
both_noatm = L[["scene_state_noatm", "exp_state"]].dropna()
print(f"\nScene vs experiencer state at the same location: rho = {spearmanr(both.scene_state, both.exp_state).correlation:+.3f} (n = {len(both)})")
print(f"Scene (without atmosphere) vs experiencer: rho = {spearmanr(both_noatm.scene_state_noatm, both_noatm.exp_state).correlation:+.3f} (n = {len(both_noatm)})")
print(f"Share of locations where the two have opposite signs: "
      f"{mw.fmt_pct(int((np.sign(both.scene_state) * np.sign(both.exp_state) < 0).sum()), len(both))}")

scene_state        n =  5265  mean -0.140  sd 0.754  share < 0: 46.1%  share > 0: 33.7%  r(log words) = +0.016
exp_state          n =  3790  mean -0.409  sd 0.792  share < 0: 60.3%  share > 0: 19.3%  r(log words) = -0.015
scene_state_noatm  n =  3882  mean +0.234  sd 0.782  share < 0: 27.6%  share > 0: 53.4%  r(log words) = +0.091

Scene vs experiencer state at the same location: rho = +0.636 (n = 3240)
Scene (without atmosphere) vs experiencer: rho = +0.389 (n = 2204)
Share of locations where the two have opposite signs: 218/3240 = 6.7% [95% CI 5.9–7.6]


## M4 — Transitions available for state-change analysis

A transition is an explicit connection from one coded location to another in the same dream, joined on the full key `(post_id, location_id)`. A list-order pair (consecutive locations in the extracted order) is the fallback for dreams without connections.

In [8]:
keep = ["post_id", "location_id", "scene_state", "exp_state", "vertical_level", "author"]
Lk = L[keep]
T = (C.merge(Lk.add_suffix("_from").rename(columns={"post_id_from": "post_id", "location_id_from": "from_location_id"}),
             on=["post_id", "from_location_id"], how="left")
      .merge(Lk.add_suffix("_to").rename(columns={"post_id_to": "post_id", "location_id_to": "to_location_id"}),
             on=["post_id", "to_location_id"], how="left"))
matched = T.scene_state_from.notna() | T.exp_state_from.notna() | T.author_from.notna()
print(f"{len(C):,} connections; both ends match a coded location: "
      f"{mw.fmt_pct(int((T.author_from.notna() & T.author_to.notna()).sum()), len(C))}")
for a, b, lab in (("scene_state_from", "scene_state_to", "scene state at both ends"),
                  ("exp_state_from", "exp_state_to", "experiencer state at both ends"),
                  ("exp_state_from", "scene_state_to", "experiencer state before, scene state after"),
                  ("scene_state_from", "exp_state_to", "scene state before, experiencer state after")):
    ok = T[a].notna() & T[b].notna()
    print(f"  {lab:46s} {int(ok.sum()):5d} transitions in {T.loc[ok, 'post_id'].nunique():4d} dreams")

Ls = L.sort_values(["post_id", "list_index"])
nxt = Ls.groupby("post_id").shift(-1)
pairs_lo = pd.DataFrame({"post_id": Ls.post_id, "scene_from": Ls.scene_state, "scene_to": nxt.scene_state,
                         "exp_from": Ls.exp_state, "exp_to": nxt.exp_state}).dropna(subset=["post_id"])
ok = pairs_lo.scene_from.notna() & pairs_lo.scene_to.notna()
print(f"\nList-order pairs with scene state at both ends: {int(ok.sum())} in {pairs_lo.loc[ok, 'post_id'].nunique()} dreams")
ok = pairs_lo.exp_from.notna() & pairs_lo.exp_to.notna()
print(f"List-order pairs with experiencer state at both ends: {int(ok.sum())} in {pairs_lo.loc[ok, 'post_id'].nunique()} dreams")

ok = T.scene_state_from.notna() & T.scene_state_to.notna()
au = T.loc[ok].groupby("author_from").post_id.nunique()
print(f"\nAuthors with transitions (scene state at both ends) in >= 2 dreams: {int((au >= 2).sum())}")

4,590 connections; both ends match a coded location: 4565/4590 = 99.5% [95% CI 99.2–99.6]
  scene state at both ends                        2513 transitions in  937 dreams
  experiencer state at both ends                  1726 transitions in  720 dreams
  experiencer state before, scene state after     1823 transitions in  740 dreams
  scene state before, experiencer state after     1872 transitions in  793 dreams



List-order pairs with scene state at both ends: 3141 in 1017 dreams
List-order pairs with experiencer state at both ends: 1967 in 773 dreams

Authors with transitions (scene state at both ends) in >= 2 dreams: 122


## M5 — Choices and influences: coverage only

**Choices** are what the person does: the interaction types at a location (social, task, transaction, search, navigation, observation, escape, conflict) and the mode and direction of each movement. **Influences** are the beings present at a location, by demeanor. Counts only; no association with state is computed here.

In [9]:
print("Transit mode on connections (the person's way of moving):")
print(C.transit_mode.value_counts().to_string())
print("\nDirection of connections:")
print(C.direction.value_counts().to_string())
inf = T.transit_mode.ne("not_mentioned") & T.scene_state_from.notna() & T.scene_state_to.notna()
print(f"\nTransitions with an informative transit mode and scene state at both ends: {int(inf.sum())}")

print("\nInteraction types (the person's acts at a location):")
print(I.interaction_type.value_counts().to_string())
acts = I.groupby(["post_id", "location_id"]).interaction_type.agg(set)
Lact = L.set_index(["post_id", "location_id"]).join(acts.rename("acts"))
has_act = Lact.acts.notna()
print(f"\nLocations with at least one coded act: {mw.fmt_pct(int(has_act.sum()), len(Lact))}")
print(f"  ... and a scene state: {int((has_act & Lact.scene_state.notna()).sum())}; and an experiencer state: {int((has_act & Lact.exp_state.notna()).sum())}")

DEM = {"helpful": "benign", "friendly": "benign", "hostile": "hostile", "threatening": "hostile", "unfriendly": "hostile",
       "neutral": "neutral", "indifferent": "neutral", "watching": "neutral", "confusing": "neutral"}
mw.check_values("demeanor", DEM)
E["dem_class"] = E.demeanor.map(DEM)
pres = E.dropna(subset=["dem_class"]).groupby(["post_id", "location_id"]).dem_class.agg(set)
Lact = Lact.join(pres.rename("beings"))
for k in ("benign", "hostile", "neutral"):
    m = Lact.beings.apply(lambda s: isinstance(s, set) and k in s)
    print(f"Locations with a {k:7s} being present: {int(m.sum()):5d}; with a scene state: {int((m & Lact.scene_state.notna()).sum()):5d}; "
          f"with an experiencer state: {int((m & Lact.exp_state.notna()).sum()):5d}")

Transit mode on connections (the person's way of moving):
transit_mode
directed_active    2531
passive             518
not_mentioned       516
wandering           363
fleeing             355
instant             171
drifting             69
struggle             67

Direction of connections:
direction
horizontal        2678
unknown            687
up                 473
down               408
not_applicable     186
diagonal_up         79
diagonal_down       79

Transitions with an informative transit mode and scene state at both ends: 2320

Interaction types (the person's acts at a location):
interaction_type
observation    1310
navigation     1219
task            811
social          793
escape          751
search          467
conflict        338
transaction     218
other           149



Locations with at least one coded act: 4096/8685 = 47.2% [95% CI 46.1–48.2]
  ... and a scene state: 3068; and an experiencer state: 2703


Locations with a benign  being present:   410; with a scene state:   332; with an experiencer state:   306
Locations with a hostile being present:   742; with a scene state:   663; with an experiencer state:   627
Locations with a neutral being present:  1019; with a scene state:   866; with an experiencer state:   741


## M6 — Added 2026-10-06: what the experiencer treats as good

The owner's principle: good and light are as the experiencer sees them (*Heaven and Hell* §§35, 429, 547, 584). Three things are kept apart:
- **Q:** the place's quality, which is the scene state without atmosphere;
- **E:** the experiencer's evaluation, the coded affect;
- **revealed good:** what the person moves toward on purpose and what he or she flees.

Counts only. No relation between choices and state change is computed.

In [10]:
both = L[L.scene_state_noatm.notna() & L.exp_state.notna()]
print(f"Locations with place quality (Q) and evaluation (E): {len(both)}")
print(pd.crosstab(np.sign(both.scene_state_noatm), np.sign(both.exp_state), rownames=["sign of Q"], colnames=["sign of E"]))
pos_in_poor = both[(both.exp_state > 0) & (both.scene_state_noatm < 0)]
neg_in_good = both[(both.exp_state < 0) & (both.scene_state_noatm > 0)]
strict = neg_in_good[neg_in_good.light.eq("bright_natural") | neg_in_good.cleanliness.eq("clean") | neg_in_good.state.eq("new")]
print(f"\nPositive evaluation in a poor-quality place: {len(pos_in_poor)} locations in {pos_in_poor.post_id.nunique()} dreams")
print(f"Negative evaluation in a good-quality place: {len(neg_in_good)} locations in {neg_in_good.post_id.nunique()} dreams; "
      f"in clearly bright, clean or new places: {len(strict)}")

kq = L[["post_id", "location_id", "scene_state_noatm"]]  # connections already carry the author
TQ = (C.merge(kq.rename(columns={"location_id": "from_location_id", "scene_state_noatm": "q_from"}), on=["post_id", "from_location_id"], how="left")
       .merge(kq.rename(columns={"location_id": "to_location_id", "scene_state_noatm": "q_to"}),
              on=["post_id", "to_location_id"], how="left"))
okq = TQ.q_from.notna() & TQ.q_to.notna()
print(f"\nTransitions with Q at both ends: {int(okq.sum())} in {TQ.loc[okq, 'post_id'].nunique()} dreams")
for mode in ("directed_active", "fleeing"):
    m = okq & TQ.transit_mode.eq(mode)
    print(f"  {mode:16s} {int(m.sum()):4d} transitions in {TQ.loc[m, 'post_id'].nunique():4d} dreams")
appr = okq & TQ.transit_mode.eq("directed_active")
print(f"Dreams with >= 2 purposeful moves (Q at both ends): {int((TQ[appr].groupby('post_id').size() >= 2).sum())}")
au = TQ[appr | (okq & TQ.transit_mode.eq("fleeing"))].groupby("author").post_id.nunique()
print(f"Authors with purposeful or flight moves (Q at both ends) in >= 2 dreams: {int((au >= 2).sum())}")

Locations with place quality (Q) and evaluation (E): 2204
sign of E  -1.0   0.0   1.0
sign of Q                  
-1.0        551    70    35
 0.0        280    92    54
 1.0        447   309   366

Positive evaluation in a poor-quality place: 35 locations in 32 dreams
Negative evaluation in a good-quality place: 447 locations in 295 dreams; in clearly bright, clean or new places: 108



Transitions with Q at both ends: 1522 in 663 dreams
  directed_active   896 transitions in  461 dreams
  fleeing           125 transitions in   92 dreams
Dreams with >= 2 purposeful moves (Q at both ends): 208
Authors with purposeful or flight moves (Q at both ends) in >= 2 dreams: 47


## Findings — measurement

**Coverage.**
- A scene state can be computed for 60.6% of locations; 35.4% have at least two indicators.
- An experiencer state can be computed for 43.6% of locations. In practice it is the coded affect: somatic responses are coded at 1.6% of locations, so only 1.4% have two indicators.
- Both states are available at 37.3% of locations.

**Do the indicators form one scene state?** Yes, by the criteria fixed in advance for coherence and length:
- median pairwise Spearman correlation 0.365 over 21 pairs; every item–rest correlation positive (0.389–0.762);
- the composites do not track narrative length (r = +0.016 for the scene state, −0.015 for the experiencer state).

**The pre-specified scaling check failed, for a diagnosed reason.** Dimension 1 of the homogeneity analysis ordered atmosphere and cleanliness against the scoring (rank correlations −0.17 and −0.89). Post hoc diagnosis (M2b):
- The first two dimensions are almost equally strong (discrimination 0.084 vs 0.098), so which one comes first depends on the random start. Four of six starts reproduce the scoring's order (atmosphere +0.54 to +0.84, cleanliness +0.89); two do not.
- With categories of fewer than 40 locations set aside, one dimension dominates (0.098 vs 0.083), and it orders every indicator as the scoring does (rank correlations +0.63 to +1.00).
- The category rest-score check agrees (+0.84 to +0.93; light temperature +0.50, where neutral and warm light co-occur with equally good scenes).

The scoring is kept unchanged. Because the check that was fixed in advance did not pass as written, every later test is also run on atmosphere alone and on the scene state without atmosphere, so that no conclusion depends on the composite alone.

**What the categories say.** Some atmosphere labels do not travel with bad scenes on the other indicators: "chaotic" (rest score +0.29), "wrong" (+0.25) and "uncomfortable" (+0.01). "Worn" places (−0.71) behave like decaying ones, and "maintained" (−0.03) is neutral. These are descriptions of how the coder used the labels; the scoring was not adjusted to them.

**Scene and experiencer.** The two states are strongly coupled at the same location (ρ = 0.636, n = 3,240), partly because they are often coded from the same sentence. Without atmosphere the coupling is 0.389. They have opposite signs at 6.7% of locations.

**What is available for state-change analysis.**
- **Transitions.** 2,513 explicit transitions have a scene state at both ends (937 dreams), and 1,726 have an experiencer state at both ends (720 dreams). Cross-lagged pairs: 1,823 and 1,872. List-order pairs add a fallback: 3,141 and 1,967.
- **Choices.** 2,320 of these transitions also have an informative movement mode: directed_active 2,531 connections overall, passive 518, wandering 363, fleeing 355, instant 171, drifting 69, struggle 67. Acts are coded at 47.2% of locations.
- **Influences.** A benign being is present at 410 locations (306 with an experiencer state) and a hostile one at 742 (627).
- **Persons.** 122 authors have transitions in at least two dreams.

**Verdict.** The scene and experiencer states are usable as measures of state for change analysis, with the caveats above: same-source coding, inferred ratings (κ 0.75 for atmosphere, 0.67 for affect where both coders rated), and prompt priming.